# SongoFish -- entrainement sur Google Colab

Un des providers de calcul du projet (`songo_ai.cloud`). Le meme code
tourne en `local` sur une machine Windows ; ici "local" = la VM Colab,
avec `SONGO_DATA_ROOT` pointe sur Google Drive (datasets + checkpoints
persistants, Colab lui-meme est ephemere).

Voir `docs/trainer/provider-architecture.md`.

Pre-requis : le dossier `SongoFish/` doit exister sur le Drive du compte,
avec `datasets/` et `checkpoints/` dedans.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/SongoFish'  # ajuster si le dossier partage a un autre chemin
import os
assert os.path.isdir(DRIVE_ROOT), f"Dossier introuvable : {DRIVE_ROOT} -- verifier le montage/partage Drive"
print('Contenu :', os.listdir(DRIVE_ROOT))

In [ ]:
# Code du projet : clone direct depuis GitHub (depot prive -> demande un
# token d'acces personnel si `git clone` anonyme echoue)
!git clone https://github.com/GlennEriss/songo-fish.git /content/songo-fish
%cd /content/songo-fish
!pip install -q -e ".[train,perf]"  # perf/numba : requis par les tournois (moteur rapide)

In [ ]:
# Verifie GPU/CPU -- rappel du projet (session locale) : ce reseau est si
# petit que le GPU n'apporte souvent aucun gain (voire ralentit), a
# mesurer ici avant de forcer device='cuda'
import torch
print('CUDA disponible:', torch.cuda.is_available())

In [ ]:
import os
os.environ['SONGO_PROVIDER'] = 'local'
os.environ['SONGO_DATA_ROOT'] = DRIVE_ROOT   # datasets/ et checkpoints/ sur le Drive
os.environ.setdefault('SONGO_DEVICE', 'cpu') # ce reseau est minuscule, le GPU n'aide pas

from songo_ai.cloud import BuildSpec, TrainSpec, MatchSpec, load_config
from songo_ai.cloud.providers import LocalProvider

provider = LocalProvider(load_config())
print(load_config())


In [ ]:
# Entrainement from scratch + enregistrement dans le registre (sur Drive)
DATASET = 'datasets/dataset_v005_400k'   # chemin logique sous DRIVE_ROOT
VERSION = '0.2.1'

result = provider.run_train(TrainSpec(
    version=VERSION,
    dataset_name=DATASET,
    epochs=150,
    early_stopping_patience=15,
    notes='Colab (provider local, data_root=Drive)',
))
result


In [ ]:
# Tournoi challenger vs champion actuel -- AVANT toute promotion
match = provider.run_tournament(MatchSpec(agent_a=VERSION, agent_b='champion', num_games=200))
print(match)

# Promotion explicite apres revue (jamais automatique) :
# from songo_ai.model import promote_version
# from pathlib import Path
# promote_version(VERSION, registry_path=Path(DRIVE_ROOT) / 'checkpoints' / 'registry.json')
